# SimCLR Self-Supervised Learning on Tiny ImageNet

This notebook runs the complete SimCLR self-supervised learning pipeline on Tiny ImageNet (200 classes, 64×64 images) using Kaggle's GPU acceleration.

## Overview
- **SimCLR Pretraining**: Learn visual representations without labels using contrastive learning
- **Supervised Baseline**: Train supervised ResNet-18 for comparison
- **Linear Evaluation**: Freeze encoder, train linear classifier
- **Evaluation & Visualization**: Compare results and visualize learned features

**Training Time**: ~2-3 days on Kaggle GPU  
**Expected Results**: ~80-85% accuracy on Tiny ImageNet test set

## Section 1: Clone and Setup Project Repository

Clone the MPA_Project repository from GitHub and set up the working environment.

In [ ]:
!git clone https://github.com/luna-droid-0206/MPA_Project
%cd MPA_Project
!ls

## Section 2: Configure Data Paths for Kaggle Environment

Set up paths for Kaggle's input/output directories and configure the dataset location.

In [ ]:
import os
import shutil

# Kaggle paths
KAGGLE_INPUT_PATH = '/kaggle/input/tiny-imagenet-200'
KAGGLE_OUTPUT_PATH = '/kaggle/working'
PROJECT_DATA_PATH = './data/tiny-imagenet-200'

# Create data directory if it doesn't exist
os.makedirs('./data', exist_ok=True)
os.makedirs('./checkpoints', exist_ok=True)
os.makedirs('./results', exist_ok=True)

print("✓ Directory structure created")
print(f"✓ Kaggle Input Path: {KAGGLE_INPUT_PATH}")
print(f"✓ Kaggle Output Path: {KAGGLE_OUTPUT_PATH}")
print(f"✓ Project Data Path: {PROJECT_DATA_PATH}")

## Section 3: Download and Extract Tiny ImageNet Dataset

**Important**: You need to add the Tiny ImageNet dataset to your Kaggle notebook first!

### How to Add Tiny ImageNet Dataset:
1. Go to: https://www.kaggle.com/datasets/akash2sharma/tiny-imagenet
2. Click "Add Data" in your notebook
3. Search for "Tiny ImageNet" or "tiny-imagenet-200"
4. Click "Add" to attach it to this notebook

The dataset will be automatically available in `/kaggle/input/tiny-imagenet-200/`

In [ ]:
# Verify and link Tiny ImageNet dataset from Kaggle input
import os
from pathlib import Path

KAGGLE_TINY_IMAGENET = '/kaggle/input/tiny-imagenet-200'
PROJECT_TINY_IMAGENET = './data/tiny-imagenet-200'

# Check if dataset exists in Kaggle input
if os.path.exists(KAGGLE_TINY_IMAGENET):
    print(f"✓ Found Tiny ImageNet at {KAGGLE_TINY_IMAGENET}")
    
    # Create symbolic link if not already exists
    if not os.path.exists(PROJECT_TINY_IMAGENET):
        try:
            os.symlink(KAGGLE_TINY_IMAGENET, PROJECT_TINY_IMAGENET)
            print(f"✓ Linked dataset to {PROJECT_TINY_IMAGENET}")
        except Exception as e:
            print(f"Could not create symlink, copying instead...")
            shutil.copytree(KAGGLE_TINY_IMAGENET, PROJECT_TINY_IMAGENET)
    
    # Verify dataset structure
    train_dir = os.path.join(PROJECT_TINY_IMAGENET, 'train')
    val_dir = os.path.join(PROJECT_TINY_IMAGENET, 'val')
    wnids_file = os.path.join(PROJECT_TINY_IMAGENET, 'wnids.txt')
    
    if os.path.exists(train_dir):
        num_classes = len(os.listdir(train_dir))
        print(f"✓ Training set: {num_classes} classes")
    
    if os.path.exists(val_dir):
        print(f"✓ Validation set: {len(os.listdir(val_dir))} images")
    
    if os.path.exists(wnids_file):
        print(f"✓ Class IDs file (wnids.txt) found")
    
    print("\n✓ Tiny ImageNet dataset is ready!")
else:
    print(f"⚠️ Warning: Tiny ImageNet not found at {KAGGLE_TINY_IMAGENET}")
    print("Make sure to add 'Tiny ImageNet' dataset to this notebook!")
    print("Go to: https://www.kaggle.com/datasets/akash2sharma/tiny-imagenet")

## Section 4: Run SimCLR Self-Supervised Pretraining

Train the SimCLR encoder on unlabeled data using contrastive learning.

**Training Details:**
- Duration: ~24-48 hours on Kaggle GPU
- Batch Size: 512
- Epochs: 300
- Optimizer: SGD with momentum
- LR Scheduler: Cosine annealing
- Mixed Precision: Enabled (FP16 + FP32)

This learns visual representations without using any labels!

In [ ]:
!python train_simclr.py --epochs 100 --batch_size 512

## Section 5: Run Supervised Baseline Training

Train a supervised ResNet-18 from scratch for comparison.

**Training Details:**
- Duration: ~2-4 hours on Kaggle GPU
- Batch Size: 512
- Epochs: 100
- Optimizer: SGD with momentum
- LR Scheduler: Cosine annealing
- Mixed Precision: Enabled

This trains with labels directly. We'll compare its accuracy with SimCLR's linear evaluation.

In [ ]:
!python train_supervised.py --epochs 50 --batch_size 512

## Section 6: Run Linear Evaluation Protocol

Freeze the pretrained SimCLR encoder and train only a linear classifier on top.

**Training Details:**
- Duration: ~1-2 hours on Kaggle GPU
- Batch Size: 512
- Epochs: 100
- Optimizer: SGD (only updates linear layer)
- LR Scheduler: Cosine annealing
- Mixed Precision: Enabled

**Key Insight**: If SimCLR learns good representations, the linear eval accuracy should exceed the supervised baseline by 5-10%! This proves the benefit of self-supervised pretraining.

In [ ]:
!python train_linear.py --epochs 50 --batch_size 512

## Section 7: Evaluate All Trained Models

Compare the performance of all trained models side-by-side.

**Expected Results:**
- Supervised Baseline: ~75-78% accuracy
- SimCLR + Linear Eval: ~80-85% accuracy ✨
- Improvement: +5-10% from self-supervised pretraining

In [ ]:
!python evaluate.py

## Section 8: Visualize Learned Feature Representations

Generate t-SNE visualizations of learned embeddings to understand how well the features cluster.

**What to Look For:**
- SimCLR features should show clear class clustering
- Points of the same class should be close together
- Different classes should be well-separated
- This visualizes the quality of learned representations!

In [ ]:
!python visualize.py

## Summary & Results

Congratulations! 🎉 You've successfully completed the SimCLR self-supervised learning pipeline on Tiny ImageNet!

### What You Accomplished:
✅ Cloned the MPA_Project repository  
✅ Configured Kaggle environment paths  
✅ Set up Tiny ImageNet dataset (200 classes, 64×64 images)  
✅ Ran SimCLR self-supervised pretraining (300 epochs)  
✅ Trained supervised baseline for comparison (100 epochs)  
✅ Performed linear evaluation on pretrained encoder (100 epochs)  
✅ Evaluated all models and compared results  
✅ Visualized learned feature representations  

### Expected Insights:
- **Self-Supervised Learning Works!** SimCLR should beat supervised baseline by 5-10%
- **Better Representations** = Improved accuracy with frozen pretrained encoder
- **Scalability** = Works effectively on larger dataset (Tiny ImageNet vs CIFAR-10)

### Outputs Generated:
- `checkpoints/simclr_pretrained_tiny_imagenet.pth` - Pretrained encoder
- `checkpoints/supervised_tiny_imagenet.pth` - Supervised baseline
- `checkpoints/linear_eval_tiny_imagenet.pth` - Linear eval classifier
- `results/*.png` - t-SNE visualizations
- `evaluation_results.json` - Performance metrics

### Key Hyperparameters Used:
- **Batch Size**: 512 (for better GPU utilization)
- **Optimizer**: SGD with momentum=0.9
- **LR Scheduler**: Cosine annealing
- **Mixed Precision**: Enabled (2-3× speedup)
- **Data Workers**: 8 (parallel loading)

### Download Results from Kaggle:
In Kaggle notebook, all outputs are saved to `/kaggle/working/`. You can download:
- Trained models from `checkpoints/`
- Visualizations from `results/`
- Evaluation metrics from `checkpoints/evaluation_results.json`

## 📚 Resources & Links

### Tiny ImageNet Dataset
- **Official Kaggle Dataset**: https://www.kaggle.com/datasets/akash2sharma/tiny-imagenet
- **Stanford CS231N**: http://cs231n.stanford.edu/tiny-imagenet-200.zip
- **Mirror Download**: http://cs231n.stanford.edu/tiny-imagenet-200.zip

### SimCLR & Self-Supervised Learning
- **SimCLR Paper**: [A Simple Framework for Contrastive Learning](https://arxiv.org/abs/2002.05709)
- **Official SimCLR GitHub**: https://github.com/google-research/simclr
- **Self-Supervised Learning Survey**: https://arxiv.org/abs/2006.07733

### Project Repository
- **GitHub**: https://github.com/luna-droid-0206/MPA_Project
- **MPA_Project**: Self-Supervised Learning Implementation

### PyTorch Documentation
- **Mixed Precision Training**: https://pytorch.org/docs/stable/notes/amp_examples.html
- **Learning Rate Schedulers**: https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
- **DataLoader**: https://pytorch.org/docs/stable/data.html

### Troubleshooting
If you encounter issues, refer to the MIGRATION_GUIDE.md in the repository for detailed troubleshooting steps.

---

**Training Time Estimate**: ~2-3 days total on Kaggle GPU  
**GPU Required**: For optimal training, use Kaggle's P100 or better GPU accelerator  
**Disk Space**: ~15GB for dataset + ~5GB for checkpoints

**Happy Learning!** 🚀